# 🧠 Entrenamiento de Modelos IA para Clasificación de Tickets
**Proyecto Soporte Tickets**

¡Bienvenido al motor de Inteligencia Artificial de nuestra plataforma! Este notebook documenta paso a paso el proceso de entrenamiento de los modelos de Machine Learning encargados de clasificar automáticamente la **Categoría** y la **Prioridad** de los tickets de soporte entrantes.

Hemos estructurado este análisis utilizando la metodología **CRISP-DM** para garantizar un flujo de trabajo ordenado y profesional.

---

## 1. Comprensión del Negocio (Business Understanding)
El objetivo principal de este módulo es reducir el tiempo de triaje (clasificación manual) en nuestra mesa de ayuda. Nuestro sistema web en Django recibe tickets de usuarios, y para agilizar la respuesta, la IA debe predecir a qué categoría pertenece el problema y qué nivel de urgencia tiene, delegando automáticamente el ticket al técnico correspondiente.

## 2. Preparación del Entorno
Comenzamos importando las librerías necesarias para la manipulación de datos (`pandas`), las herramientas de Machine Learning (`scikit-learn`) y la persistencia de los modelos (`joblib`).

*Nota de arquitectura:* Este script se ejecuta desde la carpeta `machine_learning`, por lo que las rutas dinámicas (`BASE_DIR`) buscarán los datos en el directorio local `dataset/` y exportarán los modelos listos para producción a la carpeta `modelos/`, donde posteriormente serán consumidos por nuestro backend en Django.

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

import joblib
import os

# ============================================================
# 1. CONFIGURACIÓN DE RUTAS
# ============================================================

BASE_DIR = os.path.dirname(os.path.abspath(__file__))

DATASET_PATH = os.path.join(
    BASE_DIR,
    'dataset',
    'tickets.csv'
)

MODELOS_DIR = os.path.join(
    BASE_DIR,
    'modelos'
)

os.makedirs(
    MODELOS_DIR,
    exist_ok=True
)

In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

import joblib
import os

# ============================================================
# 1. CONFIGURACIÓN DE RUTAS
# ============================================================

BASE_DIR = os.path.dirname(os.path.abspath(__file__))

DATASET_PATH = os.path.join(
    BASE_DIR,
    'dataset',
    'tickets.csv'
)

MODELOS_DIR = os.path.join(
    BASE_DIR,
    'modelos'
)

os.makedirs(
    MODELOS_DIR,
    exist_ok=True
)

## 3. Comprensión de los Datos (Data Understanding)
Antes de enseñar a la máquina, necesitamos saber con qué estamos trabajando. En esta fase cargamos nuestro historial de tickets (el archivo `tickets.csv`) y exploramos su estructura base: cuántos registros tenemos y qué columnas conforman nuestro conocimiento previo.

In [ ]:
# ============================================================
# 2. CARGAR DATASET
# ============================================================

df = pd.read_csv(DATASET_PATH)

print("\n" + "=" * 60)
print("DATASET CARGADO")
print("=" * 60)

print(df.head())

print("\nColumnas del dataset:")
print(df.columns.tolist())

print("\nCantidad de registros:")
print(len(df))

## 4. Preparación de los Datos (Data Preparation)
Los modelos de Machine Learning y el texto vacío (valores nulos o `NaN`) no se llevan bien. En esta etapa de limpieza, rellenamos cualquier espacio vacío.

Además, aplicamos *Feature Engineering* (Ingeniería de Características): en lugar de analizar el "asunto" y la "descripción" por separado, los fusionamos en una sola columna llamada `texto`. Esto le da al modelo todo el contexto semántico necesario en un solo bloque de información para mejorar la asertividad de sus predicciones.

In [ ]:
# ============================================================
# 3. LIMPIAR DATOS
# ============================================================

df['asunto'] = df['asunto'].fillna('')
df['descripcion'] = df['descripcion'].fillna('')
df['categoria'] = df['categoria'].fillna('')
df['prioridad'] = df['prioridad'].fillna('')

# ============================================================
# 4. UNIR ASUNTO + DESCRIPCIÓN
# ============================================================

df['texto'] = (
    df['asunto']
    + ' '
    + df['descripcion']
)

## 5. Modelado y Evaluación (Modeling & Evaluation)
Aquí es donde ocurre la magia. Hemos encapsulado el proceso de entrenamiento en una función robusta `entrenar_objetivo`. Esta función toma nuestra columna de texto y la variable que queremos adivinar (sea la categoría o la prioridad) y realiza lo siguiente:

1. **División (Train/Test Split):** Separa los datos guardando un 20% exclusivamente para poner a prueba el modelo, asegurando que la proporción de clases se mantenga (`stratify=y`).
2. **Vectorización (TF-IDF):** Transforma el texto humano en una matriz matemática, dándole mayor peso a las palabras relevantes y menos a las comunes.
3. **Entrenamiento Competitivo:** Entrena tres modelos distintos de la librería `scikit-learn`:
   - *Naive Bayes:* Rápido y excelente para texto base.
   - *Regresión Logística:* Confiable y con buena calibración de probabilidades.
   - *Máquinas de Vectores de Soporte (SVM):* Muy potente para encontrar fronteras de decisión complejas.
4. **Evaluación:** Compara los 3 modelos utilizando métricas clave (Accuracy, Precision, Recall y especialmente **F1 Macro** para manejar clases desbalanceadas).
5. **Selección Automática:** Escoge el modelo ganador basándose en el mejor puntaje F1.

In [ ]:
# ============================================================
# FUNCIÓN PARA ENTRENAR UN OBJETIVO
# ============================================================

def entrenar_objetivo(X, y, nombre_objetivo):

    print("\n" + "=" * 60)
    print(f"ENTRENANDO: {nombre_objetivo.upper()}")
    print("=" * 60)

    print("\nCantidad por clase:")
    print(y.value_counts())

    # ========================================================
    # TRAIN / TEST
    # ========================================================

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )

    # ========================================================
    # TF-IDF
    # ========================================================

    vectorizador = TfidfVectorizer(
        lowercase=True,
        stop_words=None
    )

    X_train_tfidf = vectorizador.fit_transform(
        X_train
    )

    X_test_tfidf = vectorizador.transform(
        X_test
    )

    # ========================================================
    # FUNCIÓN PARA EVALUAR MODELOS
    # ========================================================

    def evaluar_modelo(nombre, modelo):

        modelo.fit(
            X_train_tfidf,
            y_train
        )

        predicciones = modelo.predict(
            X_test_tfidf
        )

        accuracy = accuracy_score(
            y_test,
            predicciones
        )

        precision, recall, f1, _ = (
            precision_recall_fscore_support(
                y_test,
                predicciones,
                average='macro',
                zero_division=0
            )
        )

        print("\n" + "-" * 50)
        print(nombre)
        print("-" * 50)

        print(f"Accuracy:        {accuracy:.4f}")
        print(f"Precision Macro: {precision:.4f}")
        print(f"Recall Macro:    {recall:.4f}")
        print(f"F1 Macro:        {f1:.4f}")

        print("\nREPORTE POR CLASE")

        print(
            classification_report(
                y_test,
                predicciones,
                zero_division=0
            )
        )

        print("\nMATRIZ DE CONFUSIÓN")

        print(
            confusion_matrix(
                y_test,
                predicciones
            )
        )

        return modelo, f1

    # ========================================================
    # ENTRENAR NAIVE BAYES
    # ========================================================

    modelo_nb, f1_nb = evaluar_modelo(
        "NAIVE BAYES",
        MultinomialNB()
    )

    # ========================================================
    # ENTRENAR LOGISTIC REGRESSION
    # ========================================================

    modelo_lr, f1_lr = evaluar_modelo(
        "LOGISTIC REGRESSION",
        LogisticRegression(
            max_iter=1000
        )
    )

    # ========================================================
    # ENTRENAR SVM
    # ========================================================

    modelo_svm, f1_svm = evaluar_modelo(
        "SVM",
        SVC(
            kernel='linear',
            probability=True
        )
    )

    # ========================================================
    # SELECCIONAR MEJOR MODELO
    # ========================================================

    modelos = {
        'naive_bayes': (modelo_nb, f1_nb),
        'logistic_regression': (modelo_lr, f1_lr),
        'svm': (modelo_svm, f1_svm)
    }

    mejor_nombre = max(
        modelos,
        key=lambda nombre: modelos[nombre][1]
    )

    mejor_modelo = modelos[mejor_nombre][0]

    mejor_f1 = modelos[mejor_nombre][1]

    print("\n" + "=" * 60)
    print(
        f"MEJOR MODELO PARA "
        f"{nombre_objetivo.upper()}"
    )
    print("=" * 60)

    print(f"Modelo: {mejor_nombre}")
    print(f"F1 Macro: {mejor_f1:.4f}")

    # ========================================================
    # GUARDAR MODELO + VECTORIZADOR
    # ========================================================

    paquete_modelo = {
        'modelo': mejor_modelo,
        'vectorizador': vectorizador,
        'nombre_modelo': mejor_nombre,
        'f1_macro': mejor_f1
    }

    if nombre_objetivo == 'categoria':

        nombre_archivo = 'modelo_categoria.joblib'

    else:

        nombre_archivo = 'modelo_prioridad.joblib'

    ruta_modelo = os.path.join(
        MODELOS_DIR,
        nombre_archivo
    )

    joblib.dump(
        paquete_modelo,
        ruta_modelo
    )

    print("\nModelo guardado correctamente:")
    print(ruta_modelo)

## 6. Despliegue (Deployment)
Llegamos a la fase final de CRISP-DM. Procedemos a ejecutar nuestra función dos veces: primero para enseñarle a la IA a identificar la **Categoría** del problema, y luego para que aprenda a priorizar la **Urgencia** del mismo.

Los modelos ganadores (empaquetados junto a sus respectivos vectorizadores TF-IDF) se exportarán automáticamente mediante `.joblib` hacia la carpeta `modelos/`, quedando completamente listos para ser importados por el sistema de tickets.

In [ ]:
# ============================================================
# 5. ENTRENAR CATEGORÍA
# ============================================================

entrenar_objetivo(
    df['texto'],
    df['categoria'],
    'categoria'
)

# ============================================================
# 6. ENTRENAR PRIORIDAD
# ============================================================

entrenar_objetivo(
    df['texto'],
    df['prioridad'],
    'prioridad'
)

# ============================================================
# 7. FINALIZACIÓN
# ============================================================

print("\n" + "=" * 60)
print("ENTRENAMIENTO COMPLETADO")
print("=" * 60)

print("\nModelos generados:")

print(
    os.path.join(
        MODELOS_DIR,
        'modelo_categoria.joblib'
    )
)

print(
    os.path.join(
        MODELOS_DIR,
        'modelo_prioridad.joblib'
    )
)